# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector is built in SQL inside `capstone.ipynb` (the `build_snapshot` function), one row per page per month-end snapshot t. Every feature uses data up to and including t. This notebook reads that saved result, shows the vector, and re-derives the engineered features from their base columns to check that the stored values are what the definitions say.

- **Volume and engagement (90 days):** `imp_90`, `clicks_90`, `ctr_90`, `ctr_last30`, `active_days_90`, `cv_90` (volatility of daily impressions).
- **Position (impression-weighted, sum of position divided by impressions):** `pos_90`, `pos_last30`, `pos_prev30`, `pos_change`.
- **Momentum:** `imp_last30`, `imp_prev30`, `imp_prev60`, `log_mom`, `log_mom_prev`, and the client-relative versions `log_mom_rel`, `log_mom_prev_rel`.
- **Age:** `page_age_days`.
- **Categorical handling and fills:** the main model uses no categorical fields. Gradient boosting takes missing values as they are; the linear comparison model uses a median fill plus a missing flag.

In [2]:
import os, json, shutil
import numpy as np
import pandas as pd
import duckdb
from sklearn.metrics import average_precision_score

# The saved snapshot files and shared outputs live in Google Drive (MyDrive/flyrank_capstone),
# so this notebook can run in its own session after capstone.ipynb has finished.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT, ON_COLAB = '/content/drive/MyDrive/flyrank_capstone', True
except ImportError:
    DRIVE_ROOT, ON_COLAB = '.', False
CACHE_DIR, EVAL_DIR, OUT_DIR = 'snap_cache', 'eval_cache', f'{DRIVE_ROOT}/outputs'
if ON_COLAB:
    assert os.path.exists(f'{DRIVE_ROOT}/snap_cache'), 'Not found in Google Drive: run capstone.ipynb first, then its last cell (copy to Drive).'
    shutil.copytree(f'{DRIVE_ROOT}/snap_cache', CACHE_DIR, dirs_exist_ok=True)      # local copy: faster than reading from Drive
    if os.path.exists(f'{DRIVE_ROOT}/eval_cache'):
        shutil.copytree(f'{DRIVE_ROOT}/eval_cache', EVAL_DIR, dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
n_files = len([f for f in os.listdir(CACHE_DIR) if f.startswith('snap_') and f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
assert n_files == 12, f'{n_files} snapshot files found, expected 12: run capstone.ipynb and its last cell (copy to Drive) first.'

LABEL_DAYS = 30
K_LIST = [20, 50]
MIN_CLIENT_PAGES_MODEL = 200
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"
UNIVERSE = f"n_elig >= {MIN_CLIENT_PAGES_MODEL} AND NOT ({SPIKE_RULE})"
PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']
mcon = duckdb.connect()

def _clean(d):
    d['snap_t'] = pd.to_datetime(d['snap_t'])
    for c in PAST_COLS:
        if pd.api.types.is_timedelta64_dtype(d[c]):
            d[c] = d[c].dt.days
    return d

def load_train():
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{CACHE_DIR}/snap_*.parquet') WHERE {UNIVERSE}").df())

def load_eval(name):
    path = f'{EVAL_DIR}/eval_{name}.parquet'
    assert os.path.exists(path), f'{path} is missing: run the build cells of capstone.ipynb first'
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{path}') WHERE {UNIVERSE}").df())

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

df = load_train()
print(f"{len(df):,} labeled rows, {df['snap_t'].nunique()} snapshots, {df['client_hash_id'].nunique()} clients, base rate {df['y'].mean():.3f}")
print(f"{len(PAST_COLS)} features in the main model")

# Re-derive engineered features from their base columns and compare with the stored values
checks = {
    'ctr_90': df['clicks_90'] / df['imp_90'],
    'log_mom': np.log((df['imp_last30'] + 1.0) / (df['imp_prev30'] + 1.0)),
    'log_mom_prev': np.log((df['imp_prev30'] + 1.0) / (df['imp_prev60'] + 1.0)),
    'pos_change': df['pos_last30'] - df['pos_prev30'],
}
rows = []
for name, recomputed in checks.items():
    both = recomputed.notna() & df[name].notna()
    rows.append({'feature': name, 'rows_compared': int(both.sum()),
                 'max_abs_difference': float((recomputed[both] - df.loc[both, name]).abs().max())})
print(pd.DataFrame(rows).to_string(index=False))
assert all(r['max_abs_difference'] < 1e-6 for r in rows), 'a stored feature does not match its definition'

# Fill plan for the linear comparison model: median fill plus a missing flag (fit on the training fold only)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median', add_indicator=True)
X_filled = imputer.fit_transform(df[PAST_COLS].astype(float))
print('matrix after fill:', X_filled.shape, '(features plus one flag per feature that has missing values)')

Mounted at /content/drive
219,258 labeled rows, 12 snapshots, 13 clients, base rate 0.102
18 features in the main model
     feature  rows_compared  max_abs_difference
      ctr_90         219258                 0.0
     log_mom         219258                 0.0
log_mom_prev         219258                 0.0
  pos_change         219206                 0.0
matrix after fill: (219258, 21) (features plus one flag per feature that has missing values)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Each feature, what it means, how missing values are handled, and whether it exists before the moment of prediction (snapshot t).

| Feature | Meaning | Missing values | Known at t? |
|---|---|---|---|
| `imp_90`, `clicks_90` | Impressions and clicks over the last 90 days | None | Yes: days in (t-90, t] |
| `ctr_90`, `ctr_last30` | Clicks divided by impressions, 90 and 30 days | None (every kept page has impressions) | Yes |
| `pos_90`, `pos_last30` | Impression-weighted average position | None | Yes |
| `pos_prev30`, `pos_change` | Position in the previous 30 days; last 30 minus previous 30 | Missing when the page had no impressions in the previous 30 days (about 2% of rows) | Yes |
| `imp_last30`, `imp_prev30`, `imp_prev60` | Impressions in three consecutive 30-day windows | None | Yes |
| `active_days_90` | Days with at least one impression | None | Yes |
| `cv_90` | Volatility: standard deviation over mean of daily impressions, 90 days | None | Yes |
| `log_mom`, `log_mom_prev` | Log ratio of consecutive 30-day windows, with +1 smoothing | None | Yes |
| `log_mom_rel`, `log_mom_prev_rel` | The same, minus the client's own log ratio over the same windows (eligible pages only) | `log_mom_prev_rel` is missing for under 1% of rows, when the client has no impressions in an earlier window | Yes |
| `page_age_days` | Days from the creation date to t | None (every kept page has a creation date at least 90 days before t) | Yes |

In [ ]:
share_missing = df[PAST_COLS].isna().mean().round(4)
print('Share missing per feature (only features with missing values):')
print(share_missing[share_missing > 0].to_string())
print('\nFeatures with no missing values:', int((share_missing == 0).sum()), 'of', len(PAST_COLS))
print('\nSummary of the feature matrix:')
print(df[PAST_COLS].describe().T[['min', '50%', 'max']].round(2).to_string())

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Four attacks on my own feature set.

1. **Label-derived columns.** The label (`y`) and its ingredients (`rel`, `rel_h1`, `rel_h2`, the forward sums) are never features. The code below asserts that none of them is in the feature list.
2. **Future windows.** Features may use only dates up to t. The strongest test is a rebuild: the features for one snapshot are rebuilt from a warehouse cut at t (no row after t is read) and compared with the stored features. The capstone notebook writes the result to a small file that is shown below.
3. **Product flags.** FlyRank's own flags (for example a health score, a priority score, an action type) are not in the warehouse and are never used as features.
4. **A feature that is too good.** Each feature is used alone as a ranking score. A feature that alone ranks the label far better than the others would be a leakage suspect, and so would a model that learns from shuffled labels. The capstone's shuffled-label control stayed near or below an AP lift of 1.0 in the validation and test months and between 1.1 and 1.2 in the training folds, far below the real model.

Windows also overlap in time, so the walk-forward evaluation drops any training snapshot whose 30-day label window would run past the evaluation date (a 30-day embargo).

In [3]:
# 1. Label-derived columns and product flags are not features
LABEL_COLS = ['y', 'rel', 'rel_h1', 'rel_h2', 'f1', 'f2']
PRODUCT_FLAGS = ['health_score', 'priority_score', 'action_type', 'refresh_priority', 'trend_direction', 'trend_pct']
feature_names = set(PAST_COLS)
assert not feature_names & set(LABEL_COLS), 'a label column is in the feature list'
assert not feature_names & set(PRODUCT_FLAGS + STATIC_COLS), 'a product flag or a static field is in the main feature list'
print('No label columns and no product flags in the main feature list.')

# 2. Future windows: result of the data-cut rebuild written by capstone.ipynb
path = os.path.join(OUT_DIR, 'leakage_test.json')
if os.path.exists(path):
    print('Data-cut rebuild test:', json.load(open(path)))
else:
    print('leakage_test.json not found: run the final-queue cell of capstone.ipynb, which writes it.')

# 3. A feature that is too good? Rank by each feature alone (best direction), mean AP lift over snapshots
rows = []
for c in PAST_COLS:
    tmp = df[['snap_t', 'y']].copy()
    s = df[c].astype(float)
    tmp['s'] = s.fillna(s.median())
    up = eval_rule(tmp, 's')['ap_lift'].mean()
    tmp['s'] = -tmp['s']
    down = eval_rule(tmp, 's')['ap_lift'].mean()
    rows.append({'feature': c, 'ap_lift_alone': round(max(up, down), 2), 'direction': 'higher' if up >= down else 'lower'})
screen = pd.DataFrame(rows).sort_values('ap_lift_alone', ascending=False)
print(screen.to_string(index=False))
print('\nSuspicious (alone above 3.0):', screen.loc[screen['ap_lift_alone'] > 3.0, 'feature'].tolist() or 'none')

# 4. Shuffled-label control on one fold (labels shuffled within each training snapshot)
from sklearn.ensemble import HistGradientBoostingClassifier
t_eval = pd.Timestamp('2026-03-31')
train = df[df['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= t_eval].reset_index(drop=True)
test = df[df['snap_t'] == t_eval].copy()
if len(train) and len(test) and train['y'].nunique() == 2:
    y_sh = train['y'].values.copy()
    rng = np.random.default_rng(0)
    for _, idx in train.groupby('snap_t').indices.items():
        y_sh[idx] = rng.permutation(y_sh[idx])
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=200,
                                       l2_regularization=1.0, early_stopping=False, random_state=0)
    m.fit(train[PAST_COLS].astype(float), y_sh)
    test['null'] = m.predict_proba(test[PAST_COLS].astype(float))[:, 1]
    print('\nShuffled-label control, fold 2026-03-31:')
    print(eval_rule(test, 'null').to_string(index=False))

No label columns and no product flags in the main feature list.
Data-cut rebuild test: {'snapshot_checked': '2026-03-31', 'stored_rows': 64669, 'rows_matched': 64669, 'feature_values_that_differ': 0, 'features_checked': 18}
         feature  ap_lift_alone direction
      pos_last30           2.18    higher
          pos_90           2.16    higher
           cv_90           2.14    higher
      pos_prev30           2.13    higher
  active_days_90           2.12     lower
          ctr_90           1.79     lower
       clicks_90           1.74     lower
         log_mom           1.70     lower
     log_mom_rel           1.68     lower
      ctr_last30           1.56     lower
log_mom_prev_rel           1.53    higher
    log_mom_prev           1.49    higher
      imp_prev60           1.47     lower
      imp_last30           1.41     lower
          imp_90           1.31     lower
      pos_change           1.28    higher
      imp_prev30           1.24     lower
   page_age_days    

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Fields I refused to use in the main model, with one line of why each.

In [4]:
excluded = pd.DataFrame([
    ('fact_content_query_90d (all fields)', 'One fixed window (2026-04-02 to 2026-06-30) that overlaps the label windows; usable only as context for the final list'),
    ('content_updated_date', 'Most values fall in May to July 2026 (looks like a bulk update), so the value at an earlier t is unknown'),
    ('last_optimized_date', 'No recorded date before April 2026, so no training support; used only as a flag, not as a feature'),
    ('is_active, client_created_date, client_updated_date', 'Export-time client status; can encode the future'),
    ('is_published, is_deleted', 'Export-time status; used only to define the page universe'),
    ('keyword_created_date, optimization_eligible_date, provider_used, model_used', 'Not profiled against the panel; left out until they are'),
    ('Static content fields (word_count, backlinks, search_volume, ...)', 'Export-time values; their missingness fingerprints clients, so they are an ablation only'),
    ('GA4 metrics', 'Optional and missing for many clients; not in the main model'),
    ('sessions_ai, ai_* fields', 'Too sparse to carry signal'),
    ('gsc_avg_position', 'Cannot be averaged correctly across days; replaced by sum of position divided by impressions'),
    ('client_hash_id, client size (n_elig)', 'Context for grouping and reporting only; keeping them out stops the model from learning which client a page belongs to'),
    ('Product flags (health score, priority score, action type)', 'Outputs of FlyRank rules, not in the warehouse; never used as inputs'),
], columns=['field', 'why it is excluded'])
print(excluded.to_string(index=False))

                                                                      field                                                                                                     why it is excluded
                                        fact_content_query_90d (all fields) One fixed window (2026-04-02 to 2026-06-30) that overlaps the label windows; usable only as context for the final list
                                                       content_updated_date               Most values fall in May to July 2026 (looks like a bulk update), so the value at an earlier t is unknown
                                                        last_optimized_date                      No recorded date before April 2026, so no training support; used only as a flag, not as a feature
                        is_active, client_created_date, client_updated_date                                                                       Export-time client status; can encode the future
                         

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.